# 01_cuDF / 04 — Benchmark: pandas vs `cudf.pandas` vs cuDF

A careful comparison of three ways to run the same DataFrame work:

| Backend | What it is |
|---|---|
| **pandas** | CPU baseline |
| **cudf.pandas** | unchanged pandas code, GPU-accelerated with CPU fallback |
| **cuDF** | native GPU DataFrame API |

**Method**
- Each backend runs in its **own Python process** (a fresh process per backend and data size), so memory and caches from one run never affect another, and `cudf.pandas` can be turned on cleanly.
- Every measurement: **warmup** runs, then the **median** of several repeats.
- The GPU is **synchronized** before each timer stops.
- **IO**, **host↔device transfer** and **compute** are reported separately.
- Data sizes from 100k to 10M rows, to show where the GPU starts to pay off.
- Environment details are logged first and saved next to the results.

> Restart the kernel before running (don't run this in a kernel where `%load_ext cudf.pandas` was used). The full run takes about **10 minutes** on a T4, most of it waiting for pandas.

## 1. Environment

In [ ]:
import os, sys, json, platform, subprocess, importlib.util, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

if "cudf.pandas" in sys.modules:
    print("WARNING: cudf.pandas is loaded in this kernel. Restart the kernel for a clean benchmark.")

HAS_CUDF = importlib.util.find_spec("cudf") is not None

def cpu_model():
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"

def ram_gb():
    try:
        with open("/proc/meminfo") as f:
            return round(int(f.readline().split()[1]) / 1e6, 1)
    except OSError:
        return None

env = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "cpu": cpu_model(),
    "cpu_cores": os.cpu_count(),
    "ram_gb": ram_gb(),
}
if HAS_CUDF:
    import cudf, cupy as cp
    props = cp.cuda.runtime.getDeviceProperties(0)
    rt, drv = cp.cuda.runtime.runtimeGetVersion(), cp.cuda.runtime.driverGetVersion()
    env.update({
        "cudf": cudf.__version__,
        "cupy": cp.__version__,
        "gpu": props["name"].decode(),
        "gpu_mem_gb": round(props["totalGlobalMem"] / 1e9, 1),
        "compute_capability": f"{props['major']}.{props['minor']}",
        "cuda_runtime": f"{rt // 1000}.{(rt % 1000) // 10}",
        "cuda_driver": f"{drv // 1000}.{(drv % 1000) // 10}",
    })
BACKENDS = ["pandas"] + (["cudf.pandas", "cudf"] if HAS_CUDF else [])
print("Backends to run:", BACKENDS)
pd.Series(env, name="value").to_frame()

## 2. Configuration

In [ ]:
SIZES   = [100_000, 1_000_000, 5_000_000, 10_000_000]
REPEATS = 3
WARMUP  = 1
N_CUSTOMERS = 200_000

DATA_DIR, RESULTS_DIR = "data", "results"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## 3. Data

One Parquet file per size, generated on the GPU with a fixed seed. Every backend reads the same files.

In [ ]:
import random

random.seed(0)
categories = ["ELEC", "GROC", "CLOT", "HOME", "SPOR", "BOOK"]
colors, sizes = ["Red", "Blue", "Black", "White", "Green"], ["S", "M", "L", "XL"]
product_names = [f"{random.choice(categories)}-{i:04d} {random.choice(colors)} {random.choice(sizes)}" for i in range(2_000)]
cities = ["Hyderabad", "Bengaluru", "Pune", "Jaipur", "Chennai", "Delhi", "Mumbai", "Kolkata"]
start_s = np.datetime64("2026-01-01T00:00:00", "s").astype("int64")

def make_orders(n, seed=0):
    rng = cp.random.default_rng(seed)
    df = cudf.DataFrame({
        "customer_id":  rng.integers(0, N_CUSTOMERS, n).astype("int32"),
        "store_id":     rng.integers(0, 50, n).astype("int16"),
        "quantity":     rng.integers(1, 10, n).astype("int16"),
        "amount":       rng.random(n) * 500,
        "city":         cudf.Series(cities).take(rng.integers(0, len(cities), n)).reset_index(drop=True),
        "product_name": cudf.Series(product_names).take(rng.integers(0, len(product_names), n)).reset_index(drop=True),
        "ts":           cudf.Series(start_s + rng.integers(0, 180 * 86_400, n)).astype("datetime64[s]"),
    })
    df["month"] = df["ts"].dt.month
    return df

paths = {}
for n in SIZES:
    paths[n] = os.path.join(DATA_DIR, f"bench_{n}.parquet")
    if not os.path.exists(paths[n]):
        make_orders(n).to_parquet(paths[n])
    print(f"{n:>12,} rows  {os.path.getsize(paths[n]) / 1e6:8.1f} MB  {paths[n]}")

cust_path = os.path.join(DATA_DIR, "bench_customers.parquet")
seg = cudf.Series(["Consumer", "Corporate", "Home Office"])
cudf.DataFrame({
    "customer_id": cp.arange(N_CUSTOMERS, dtype="int32"),
    "segment":     seg.take(cp.random.default_rng(1).integers(0, 3, N_CUSTOMERS)).reset_index(drop=True),
}).to_parquet(cust_path)

## 4. The benchmark script

One script, three backends. For `pandas` and `cudf.pandas` it does `import pandas as xpd`; for `cudf` it does `import cudf as xpd`. The operations are written once and run unchanged on all three.

For native cuDF it also measures the **transfer** cost of moving the whole table between host and GPU memory (`from_pandas` / `to_pandas`). With `cudf.pandas` those copies happen implicitly, inside the timed operations.

In [ ]:
%%writefile bench_cudf.py
import argparse, json, statistics, time

p = argparse.ArgumentParser()
p.add_argument("--backend", choices=["pandas", "cudf", "cudf.pandas"], required=True)
p.add_argument("--data", required=True)
p.add_argument("--customers", required=True)
p.add_argument("--repeats", type=int, default=3)
p.add_argument("--warmup", type=int, default=1)
a = p.parse_args()

if a.backend == "cudf":
    import cudf as xpd
else:
    import pandas as xpd      # under `python -m cudf.pandas` this is the accelerated proxy

if a.backend == "pandas":
    def sync():
        pass
else:
    import cupy as cp
    def sync():
        cp.cuda.Device().synchronize()

records = []

def bench(op, kind, fn):
    for _ in range(a.warmup):
        fn(); sync()
    times = []
    for _ in range(a.repeats):
        sync()
        t0 = time.perf_counter()
        fn()
        sync()
        times.append(time.perf_counter() - t0)
    records.append({"op": op, "kind": kind, "median_s": statistics.median(times),
                    "min_s": min(times), "max_s": max(times)})

bench("read_parquet", "io", lambda: xpd.read_parquet(a.data))
df = xpd.read_parquet(a.data)
cust = xpd.read_parquet(a.customers)

if a.backend == "cudf":
    import pandas as pd
    pdf = pd.read_parquet(a.data)
    bench("host->device (from_pandas)", "transfer", lambda: xpd.from_pandas(pdf))
    bench("device->host (to_pandas)", "transfer", lambda: df.to_pandas())
    del pdf

ops = {
    "filter":            lambda: df[df["amount"] > 250],
    "groupby sum":       lambda: df.groupby("store_id")["amount"].sum(),
    "groupby multi-agg": lambda: df.groupby(["store_id", "month"]).agg({"amount": ["sum", "mean"], "quantity": "max"}),
    "groupby nunique":   lambda: df.groupby("store_id")["customer_id"].nunique(),
    "merge":             lambda: df.merge(cust, on="customer_id"),
    "sort 1 key":        lambda: df.sort_values("amount"),
    "sort 2 keys":       lambda: df.sort_values(["store_id", "amount"]),
    "drop_duplicates":   lambda: df.drop_duplicates(subset=["customer_id"]),
    "value_counts":      lambda: df["city"].value_counts(),
    "datetime groupby":  lambda: df.groupby(df["ts"].dt.month)["amount"].sum(),
    "str.contains":      lambda: df["product_name"].str.contains(r"Blue|Red", regex=True),
    "str.extract":       lambda: df["product_name"].str.extract(r"-(\d{4})"),
    "rolling mean":      lambda: df["amount"].rolling(100).mean(),
    "groupby cumsum":    lambda: df.groupby("store_id")["amount"].cumsum(),
}
for name, fn in ops.items():
    bench(name, "compute", fn)

print("RESULTS " + json.dumps(records))

## 5. Run

In [ ]:
def run(backend, rows):
    cmd = [sys.executable] + (["-m", "cudf.pandas"] if backend == "cudf.pandas" else []) + [
        "bench_cudf.py", "--backend", backend, "--data", paths[rows], "--customers", cust_path,
        "--repeats", str(REPEATS), "--warmup", str(WARMUP)]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-3000:])
    line = next(l for l in proc.stdout.splitlines() if l.startswith("RESULTS "))
    recs = json.loads(line[len("RESULTS "):])
    for r in recs:
        r.update(backend=backend, rows=rows)
    return recs

all_records = []
for rows in SIZES:
    for backend in BACKENDS:
        t0 = time.perf_counter()
        try:
            all_records += run(backend, rows)
            print(f"{rows:>12,} rows | {backend:<12} done in {time.perf_counter() - t0:6.1f} s")
        except Exception as e:
            print(f"{rows:>12,} rows | {backend:<12} FAILED, skipping\n{e}")

res = pd.DataFrame(all_records)
res.to_csv(os.path.join(RESULTS_DIR, "04_benchmark_cudf_vs_pandas.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "04_benchmark_env.json"), "w") as f:
    json.dump(env, f, indent=2)
print(f"\n{len(res)} measurements saved to {RESULTS_DIR}/")

## 6. Results

### Speedup at the largest size

In [ ]:
order = [b for b in ["pandas", "cudf.pandas", "cudf"] if b in res["backend"].unique()]
N_MAX = res["rows"].max()

comp = res[res["kind"] == "compute"]
big = comp[comp["rows"] == N_MAX].pivot(index="op", columns="backend", values="median_s")[order]
for b in order[1:]:
    big[f"speedup {b}"] = big["pandas"] / big[b]
big = big.sort_values(f"speedup {order[-1]}") if len(order) > 1 else big
big.round(4)

In [ ]:
colors = {"cudf.pandas": "#5B8FF9", "cudf": "#4C9F70"}
fig, ax = plt.subplots(figsize=(9, 0.42 * len(big) + 1.2))
y = np.arange(len(big))
h = 0.38
for i, b in enumerate(order[1:]):
    vals = big[f"speedup {b}"]
    ax.barh(y + (i - 0.5) * h, vals, height=h, label=b, color=colors[b])
    for yi, v in zip(y, vals):
        ax.text(v, yi + (i - 0.5) * h, f" {v:.1f}x", va="center", fontsize=7)
ax.set_yticks(y, big.index)
ax.axvline(1, color="gray", lw=1, ls="--")
ax.set_xscale("log")
ax.set_xlabel("speedup vs pandas (log scale, higher is better)")
ax.set_title(f"{N_MAX:,} rows on {env.get('gpu', 'CPU')}")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

### How speedup changes with data size

Each cell is the speedup over pandas for one operation at one data size. Values **below 1** mean the GPU was slower: at small sizes, the fixed cost of launching GPU kernels outweighs the work.

In [ ]:
def speedup_grid(backend):
    piv = comp.pivot_table(index="op", columns=["backend", "rows"], values="median_s")
    return (piv["pandas"] / piv[backend]).loc[big.index]

fig, axes = plt.subplots(1, len(order) - 1, figsize=(6 * (len(order) - 1), 0.42 * len(big) + 1.5), squeeze=False)
for ax, b in zip(axes[0], order[1:]):
    grid = speedup_grid(b)
    im = ax.imshow(np.log10(grid.values), cmap="RdYlGn", vmin=-1, vmax=2, aspect="auto")
    ax.set_xticks(range(grid.shape[1]), [f"{c:,}" for c in grid.columns], rotation=30)
    ax.set_yticks(range(grid.shape[0]), grid.index)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            ax.text(j, i, f"{grid.values[i, j]:.1f}x", ha="center", va="center", fontsize=7)
    ax.set_title(f"{b} speedup vs pandas"); ax.set_xlabel("rows")
plt.tight_layout(); plt.show()

### Scaling: time vs rows

In [ ]:
show_ops = ["groupby multi-agg", "merge", "sort 2 keys", "str.extract"]
fig, axes = plt.subplots(1, len(show_ops), figsize=(4 * len(show_ops), 3.2), sharey=False)
line_colors = {"pandas": "#E8684A", **colors}
for ax, op in zip(axes, show_ops):
    for b in order:
        d = comp[(comp["op"] == op) & (comp["backend"] == b)].sort_values("rows")
        ax.plot(d["rows"], d["median_s"], marker="o", label=b, color=line_colors[b])
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(op); ax.set_xlabel("rows")
axes[0].set_ylabel("seconds (median, log)")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Transfer vs compute (native cuDF)

How long it takes to get data **onto** the GPU and back, compared with running all 14 operations once it is there.

In [ ]:
if "cudf" in order:
    cu = res[res["backend"] == "cudf"]
    parts = cu[cu["kind"] != "compute"].pivot(index="rows", columns="op", values="median_s")
    parts["all 14 ops (compute)"] = cu[cu["kind"] == "compute"].groupby("rows")["median_s"].sum()
    parts = parts[["read_parquet", "host->device (from_pandas)", "device->host (to_pandas)", "all 14 ops (compute)"]]

    ax = parts.plot.bar(figsize=(9, 3.5), rot=0, logy=True)
    ax.set_xticklabels([f"{r:,}" for r in parts.index])
    ax.set_ylabel("seconds (median, log)"); ax.set_xlabel("rows")
    ax.set_title("cuDF: IO and transfer vs compute")
    plt.tight_layout(); plt.show()
    display(parts.round(4))

### Total time for all operations

In [ ]:
totals = comp.groupby(["rows", "backend"])["median_s"].sum().unstack()[order]
ax = totals.plot(marker="o", figsize=(7, 3.5), logx=True, logy=True, color=[line_colors[b] for b in order])
ax.set_xlabel("rows"); ax.set_ylabel("seconds for all 14 ops (log)")
ax.set_title("Total compute time")
plt.tight_layout(); plt.show()
totals.round(3)

## 7. Reading the results

Things to look for in your numbers:

- **Small data (100k rows):** GPU speedups shrink or drop below 1×. Each GPU operation has a fixed launch cost of tens of microseconds, which dominates when there is little work.
- **Large data:** sorts, joins and string operations usually gain the most. Simple reductions gain less, because pandas (NumPy) is already efficient at them.
- **cudf.pandas vs cuDF:** `cudf.pandas` is usually a bit slower than native cuDF due to proxy overhead and occasional fallbacks, but needs no code changes.
- **Transfer:** if `from_pandas` takes longer than the compute you plan to do, load data straight onto the GPU (`cudf.read_parquet`) instead of building it in pandas first.

The raw numbers are in `results/04_benchmark_cudf_vs_pandas.csv` and the environment in `results/04_benchmark_env.json`.

In [ ]:
os.remove("bench_cudf.py")
# Uncomment to delete the generated data files as well
# for p in list(paths.values()) + [cust_path]:
#     os.remove(p)

## Summary

- One script, three backends, each in a fresh process; warmup + median; GPU synced; IO, transfer and compute kept separate.
- The GPU wins grow with data size; at small sizes, fixed overheads can make it slower than pandas.
- Native cuDF is the fastest; `cudf.pandas` gets most of the way with no code changes.

**Next folder:** `02_cuVS/` — GPU vector search.